# 0. Preparation

In [ ]:
# import sys;

# !{sys.executable} -m pip install opencv-python numpy matplotlib seaborn tqdm scikit-image scikit-learn scipy

# 1. Import relevant packages and set up basic variables

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
import json

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from skimage.color import rgb2gray
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

path_template_EWS = './data/EWS-Dataset/{}'
path_template_HLC = './data/DataforYuetal/{}/HLC'
path_template_LLC = './data/DataforYuetal/{}/LLC'

IMG_SIZE = (350, 350)
SEED = 34
random.seed(SEED)
np.random.seed(SEED)

# 2. Define the functions: Load the images and calculate metrics

In [ ]:
def load_single_folder(path):
    imgs = []
    masks = []
    files = sorted([f for f in os.listdir(path) if not '_mask' in f and f.endswith('.png')])
    for f in files:
        img = cv2.imread(os.path.join(path, f))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)

        mask_f = f.replace('.png', '_mask.png')
        mask = cv2.imread(os.path.join(path, mask_f), 0)
        mask = cv2.resize(mask, IMG_SIZE)
        mask = (mask > 127).astype(np.uint8)

        imgs.append(img)
        masks.append(mask)
    return np.array(imgs), np.array(masks)

def calculate_metrics(pred_mask, gt_mask):
    p = pred_mask.flatten()
    g = gt_mask.flatten()
    prec = precision_score(g, p, zero_division=0)
    rec = recall_score(g, p, zero_division=0)
    f1 = f1_score(g, p, zero_division=0)
    intersection = np.logical_and(g, p).sum()
    union = np.logical_or(g, p).sum()
    iou = intersection / union if union != 0 else 0.0
    return {
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "iou": round(iou, 4)
    }

def print_results(title, result_dict):
    print("=" * 130)
    print(title)
    print("=" * 130)
    print(f"{'Method':<15} {'Dataset':<8} {'IoU':<10} {'F1':<10} {'Precision':<10} {'Recall':<10}")
    print("=" * 130)
    count = 0
    for key in result_dict:
        parts = key.split("_")
        
        if len(parts) == 2:
            ds_name = parts[0]
            method_display = parts[1]
            res = result_dict[key]
            print(f"{method_display:<15} {ds_name:<8} {res['iou']:<10.4f} {res['f1']:<10.4f} {res['precision']:<10.4f} {res['recall']:<10.4f}")
        
        elif len(parts) >= 3:
            ds_name = parts[0]
            clf_name = parts[1]
            ft_name = "_".join(parts[2:])
            method_display = f"{clf_name}_{ft_name}"
            res = result_dict[key]
            print(f"{method_display:<15} {ds_name:<8} {res['iou']:<10.4f} {res['f1']:<10.4f} {res['precision']:<10.4f} {res['recall']:<10.4f}")
        if count % 3 == 2:
            print("-" * 130)
        count += 1

def plot_top_samples(saved_samples, plot_title, count=3):
    valid_samples = []
    for s in saved_samples:
        img, gt_mask, pred_mask, score = s
        is_all_black = (np.sum(pred_mask) == 0)
        is_all_white = (np.sum(pred_mask) == pred_mask.size)

        if not is_all_black and not is_all_white:
            valid_samples.append(s)

    sorted_valid = sorted(valid_samples, key=lambda x: x[3], reverse=True)
    top_n = sorted_valid[:count]

    sorted_all = sorted(saved_samples, key=lambda x: x[3])
    worst_3 = sorted_all[:count]

    plt.rcParams['figure.dpi'] = 50
    plt.rcParams['savefig.dpi'] = 50
    plt.rcParams['figure.autolayout'] = True

    print(f"\n===== {plot_title} | TOP {count} BEST PREDICTIONS =====")
    for idx, (img, gt_mask, pred_mask, score) in enumerate(top_n):
        plt.figure(figsize=(6, 2)) 
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title(f"TOP {idx+1}", fontsize=14)
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(gt_mask, cmap="gray")
        plt.title("Ground Truth", fontsize=14)
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask, cmap="gray")
        plt.title(f"IoU={score:.2f}", fontsize=14)
        plt.axis("off")

        plt.show()
        plt.close('all')

    print(f"\n===== {plot_title} | WORST {count} PREDICTIONS =====")
    for idx, (img, gt_mask, pred_mask, score) in enumerate(worst_3):
        plt.figure(figsize=(6, 2))
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title(f"Worst {idx+1}", fontsize=14)
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(gt_mask, cmap="gray")
        plt.title("Ground Truth", fontsize=14)
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask, cmap="gray")
        plt.title(f"IoU={score:.2f}", fontsize=14)
        plt.axis("off")

        plt.show()
        plt.close('all')

# 3. Load the images

In [ ]:
train_ews_path = path_template_EWS.format('train')
val_ews_path = path_template_EWS.format('validation')
test_ews_path = path_template_EWS.format('test')

train_imgs_ews, train_masks_ews = load_single_folder(train_ews_path)
val_imgs_ews, val_masks_ews = load_single_folder(val_ews_path)
test_imgs_ews, test_masks_ews = load_single_folder(test_ews_path)

train_hlc_path = path_template_HLC.format('train')
val_hlc_path = path_template_HLC.format('validation')
test_hlc_path = path_template_HLC.format('test')

train_imgs_hlc, train_masks_hlc = load_single_folder(train_hlc_path)
val_imgs_hlc, val_masks_hlc = load_single_folder(val_hlc_path)
test_imgs_hlc, test_masks_hlc = load_single_folder(test_hlc_path)

train_llc_path = path_template_LLC.format('train')
val_llc_path = path_template_LLC.format('validation')
test_llc_path = path_template_LLC.format('test')

train_imgs_llc, train_masks_llc = load_single_folder(train_llc_path)
val_imgs_llc, val_masks_llc = load_single_folder(val_llc_path)
test_imgs_llc, test_masks_llc = load_single_folder(test_llc_path)

print('Done.')

# 4. Use advanced segmentation methods to compare
## 4.1 Define advanced segmentation methods

In [ ]:
def extract_green_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    lower_green = np.array([30, 40, 40])
    upper_green = np.array([90, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return mask

def watershed_seg(img, fg_ratio=0.05, morph_kernel=3):
    mask = extract_green_mask(img)
    kernel = np.ones((morph_kernel, morph_kernel), np.uint8)
    bg = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    dist = cv2.distanceTransform(bg, cv2.DIST_L2, 5)
    _, fg = cv2.threshold(dist, fg_ratio * dist.max(), 255, 0)
    fg = np.uint8(fg)

    unknown = cv2.subtract(bg, fg)
    _, markers = cv2.connectedComponents(fg)
    markers += 1
    markers[unknown == 255] = 0

    markers = cv2.watershed(cv2.cvtColor(img, cv2.COLOR_RGB2BGR), markers)

    out = np.ones_like(mask, dtype=np.uint8)
    out[markers > 1] = 0
    return out

def mser_seg(img, min_area=10, blur_kernel=3):
    mask = extract_green_mask(img)

    mser = cv2.MSER_create()
    mser.setMinArea(min_area)
    mser.setMaxArea(5000)
    regions, _ = mser.detectRegions(mask)

    out_mask = np.zeros_like(mask)
    for pts in regions:
        cv2.drawContours(out_mask, [pts], -1, 255, -1)
    out_mask = cv2.morphologyEx(out_mask, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    return 1 - (out_mask // 255)

def superpixel_seg(img, n_segments=150):
    from skimage.segmentation import slic
    mask = extract_green_mask(img)
    segments = slic(img, n_segments=n_segments, compactness=10, sigma=1)

    out = np.zeros_like(mask)

    global_mean = mask.mean()
    # Optimize in different cases
    threshold = 15 if global_mean < 10 else 80

    for lab in np.unique(segments):
        coords = np.where(segments == lab)
        mean_green = np.mean(mask[coords])
        if mean_green > threshold:
            out[coords] = 1

    kernel = np.ones((1, 1), np.uint8)
    out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, kernel, iterations=1)

    return 1 - out

## 4.2 Call the functions above with train and validation datasets and optimize the methods.

In [ ]:
def eval_advanced(imgs, masks, dataset_name="Dataset"):
    ws_saved = []
    ms_saved = []
    sp_saved = []

    ws_list = []
    ms_list = []
    sp_list = []

    for idx, (img, gt) in enumerate(zip(imgs, masks)):
        pred_ws = watershed_seg(img)
        pred_ms = mser_seg(img)
        pred_sp = superpixel_seg(img)

        s_ws = calculate_metrics(pred_ws, gt)
        s_ms = calculate_metrics(pred_ms, gt)
        s_sp = calculate_metrics(pred_sp, gt)

        ws_list.append(s_ws)
        ms_list.append(s_ms)
        sp_list.append(s_sp)

        ws_saved.append((img, gt, pred_ws, s_ws["iou"]))
        ms_saved.append((img, gt, pred_ms, s_ms["iou"]))
        sp_saved.append((img, gt, pred_sp, s_sp["iou"]))

    def average(scores):
        return {
            "iou": round(np.mean([s["iou"] for s in scores]), 4),
            "f1": round(np.mean([s["f1"] for s in scores]), 4),
            "precision": round(np.mean([s["precision"] for s in scores]), 4),
            "recall": round(np.mean([s["recall"] for s in scores]), 4)
        }

    plot_top_samples(ws_saved, f"Watershed {dataset_name}")
    plot_top_samples(ms_saved, f"MSER {dataset_name}")
    plot_top_samples(sp_saved, f"Superpixel {dataset_name}")

    return {
        "watershed": average(ws_list),
        "mser": average(ms_list),
        "superpixel": average(sp_list)
    }
tune_imgs_ews = np.concatenate([train_imgs_ews, val_imgs_ews])
tune_masks_ews = np.concatenate([train_masks_ews, val_masks_ews])

tune_imgs_hlc = np.concatenate([train_imgs_hlc, val_imgs_hlc])
tune_masks_hlc = np.concatenate([train_masks_hlc, val_masks_hlc])

tune_imgs_llc = np.concatenate([train_imgs_llc, val_imgs_llc])
tune_masks_llc = np.concatenate([train_masks_llc, val_masks_llc])

print('Evaluate EWS with TRAIN+VAL dataset.')
adv_val_ews = eval_advanced(tune_imgs_ews, tune_masks_ews, "EWS_TRAIN+VAL")
print('Evaluate HLC with TRAIN+VAL dataset.')
adv_val_hlc = eval_advanced(tune_imgs_hlc, tune_masks_hlc, "HLC_TRAIN+VAL")
print('Evaluate LLC with TRAIN+VAL dataset.')
adv_val_llc = eval_advanced(tune_imgs_llc, tune_masks_llc, "LLC_TRAIN+VAL")

adv_formatted = {
    "EWS_watershed": adv_val_ews["watershed"],
    "EWS_mser": adv_val_ews["mser"],
    "EWS_superpixel": adv_val_ews["superpixel"],

    "HLC_watershed": adv_val_hlc["watershed"],
    "HLC_mser": adv_val_hlc["mser"],
    "HLC_superpixel": adv_val_hlc["superpixel"],

    "LLC_watershed": adv_val_llc["watershed"],
    "LLC_mser": adv_val_llc["mser"],
    "LLC_superpixel": adv_val_llc["superpixel"]
}
print_results("ADVANCED SEGMENTATION - VALIDATION RESULTS (TRAIN+VAL)", adv_formatted)

## 4.3 Compare the results of test dataset

In [ ]:
adv_test_ews = eval_advanced(test_imgs_ews, test_masks_ews, "EWS_TEST")
adv_test_hlc = eval_advanced(test_imgs_hlc, test_masks_hlc, "HLC_TEST")
adv_test_llc = eval_advanced(test_imgs_llc, test_masks_llc, "LLC_TEST")

adv_test_formatted = {
    "EWS_watershed": adv_test_ews["watershed"],
    "EWS_mser": adv_test_ews["mser"],
    "EWS_superpixel": adv_test_ews["superpixel"],

    "HLC_watershed": adv_test_hlc["watershed"],
    "HLC_mser": adv_test_hlc["mser"],
    "HLC_superpixel": adv_test_hlc["superpixel"],

    "LLC_watershed": adv_test_llc["watershed"],
    "LLC_mser": adv_test_llc["mser"],
    "LLC_superpixel": adv_test_llc["superpixel"]
}

print_results("ADVANCED SEGMENTATION - TEST RESULTS", adv_test_formatted)

## 4.4 Conclusion: Advanced Segmentation Methods

### 1. Performance Overview (Highest IoU on Test Set)
- **EWS Dataset**:
  - Watershed (0.8324) > Superpixel (0.7237) > MSER (0.6267)
- **HLC Dataset**:
  - Watershed (0.7972) > Superpixel (0.6798) > MSER (0.5819)
- **LLC Dataset**:
  - Watershed (0.9178) > Superpixel (0.8303) > MSER (0.7355)

### 2. Stability & Balance (Precision vs Recall)
- **Watershed**: Most balanced across all datasets.
  - EWS: Precision=0.8661, Recall=0.9556
  - HLC: Precision=0.8350, Recall=0.9489
  - LLC: Precision=0.9416, Recall=0.9720
- **Superpixel**: High precision in suppressing shadows and non-vegetation regions, with relatively lower recall on sparse wheat.
  - EWS: Precision=0.8770, Recall=0.8047
  - HLC: Precision=0.8496, Recall=0.7766
  - LLC: Precision=0.9437, Recall=0.8729
- **MSER**: Lower and less balanced, especially on challenging datasets.
  - EWS: Precision=0.6616, Recall=0.8316
  - HLC: Precision=0.6166, Recall=0.8213
  - LLC: Precision=0.7710, Recall=0.8566

### 3. Robustness (IoU Drop from Validation to Test)
- **Watershed**: Most robust, with a moderate drop.
  - EWS: 0.8819 → 0.8324
  - HLC: 0.8626 → 0.7972
  - LLC: 0.8985 → 0.9178 (slight improvement)
- **Superpixel**: Moderate drop in performance, maintaining stable precision with adaptive green thresholding.
  - EWS: 0.7930 → 0.7237
  - HLC: 0.7560 → 0.6798
  - LLC: 0.8250 → 0.8303 (slight improvement)
- **MSER**: Largest drop in performance, especially on EWS and HLC.
  - EWS: 0.7583 → 0.6267
  - HLC: 0.7068 → 0.5819
  - LLC: 0.8029 → 0.7355

### 4. Analysis & Reasons
- **Watershed**:
  Leverages edge information and distance transform, making it robust to variations in color intensity and lighting. Performs well across dense and sparse vegetation without over-reliance on color thresholds.
- **Superpixel**:
  Uses adaptive region clustering and green-strength filtering to reduce false positives from shadows and background. Achieves high precision but may under-segment sparse wheat due to stricter vegetation criteria.
- **MSER**:
  Highly sensitive to local contrast and the quality of the green mask. Struggles in weak-green, sparse, or occluded regions, leading to frequent false negatives and unstable predictions.

Note: IoU is sensitive to target size. Smaller wheat regions may yield higher IoU with fewer correct predictions, while larger dense regions tend to have lower IoU despite good overall segmentation.

# 5. Use machine learning-based methods to compare
## 5.1 Use single feature to train the model.
### 5.1.1 Use hog as the single feature and compare between classifiers.
#### 5.1.1.1 Define basic functions

In [ ]:
PATCH_SIZE_HOG = 25
STEP_HOG = 25

def extract_patches_hog(img, mask):
    patches = []
    labels = []
    h, w = img.shape[:2]
    for y in range(0, h - PATCH_SIZE_HOG + 1, STEP_HOG):
        for x in range(0, w - PATCH_SIZE_HOG + 1, STEP_HOG):
            patch = img[y:y+PATCH_SIZE_HOG, x:x+PATCH_SIZE_HOG]
            patch_mask = mask[y:y+PATCH_SIZE_HOG, x:x+PATCH_SIZE_HOG]
            cy, cx = PATCH_SIZE_HOG // 2, PATCH_SIZE_HOG // 2
            label = patch_mask[cy, cx]
            patches.append(patch)
            labels.append(label)
    return patches, labels

def get_hog_feature(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    return hog(gray, pixels_per_cell=(8,8), cells_per_block=(2,2), visualize=False)

def build_dataset_hog(imgs, masks):
    X = []
    y = []
    for img, mask in zip(imgs, masks):
        patches, labels = extract_patches_hog(img, mask)
        for p, lb in zip(patches, labels):
            X.append(get_hog_feature(p))
            y.append(lb)
    return np.array(X), np.array(y)

def predict_image_hog(img, model, scaler):
    h, w = img.shape[:2]
    output = np.zeros((h, w), dtype=np.uint8)
    for y in range(0, h - PATCH_SIZE_HOG + 1, STEP_HOG):
        for x in range(0, w - PATCH_SIZE_HOG + 1, STEP_HOG):
            patch = img[y:y+PATCH_SIZE_HOG, x:x+PATCH_SIZE_HOG]
            feat = get_hog_feature(patch).reshape(1, -1)
            feat = scaler.transform(feat)
            pred = model.predict(feat)[0]
            output[y:y+PATCH_SIZE_HOG, x:x+PATCH_SIZE_HOG] = pred
    return output

#### 5.1.1.2 Use train and validation dataset to train and optimize the model.

In [ ]:
hog_models = {}
hog_scalers = {}
hog_val_results = {}

datasets = {
    "EWS": (train_imgs_ews, train_masks_ews, val_imgs_ews, val_masks_ews),
    "HLC": (train_imgs_hlc, train_masks_hlc, val_imgs_hlc, val_masks_hlc),
    "LLC": (train_imgs_llc, train_masks_llc, val_imgs_llc, val_masks_llc)
}

classifiers = {
    "svm": SVC(),
    "rf": RandomForestClassifier(n_estimators=50, random_state=SEED),
    "knn": KNeighborsClassifier(n_neighbors=3)
}

for ds_name in datasets:
    print(f"=== Training HOG on {ds_name} ===")
    tr_imgs, tr_masks, val_imgs, val_masks = datasets[ds_name]

    X_train, y_train = build_dataset_hog(tr_imgs, tr_masks)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    for clf_name, clf in classifiers.items():
        print(f"Training {clf_name}...")

        clf.fit(X_train, y_train)

        key = f"{ds_name}_{clf_name}_hog"
        hog_models[key] = clf
        hog_scalers[key] = scaler

        ious, f1s, pres, recs = [], [], [], []

        saved_samples = []

        for img, gt in zip(val_imgs, val_masks):
            pred_mask = predict_image_hog(img, clf, scaler)
            res = calculate_metrics(pred_mask, gt)
            ious.append(res["iou"])
            f1s.append(res["f1"])
            pres.append(res["precision"])
            recs.append(res["recall"])
            saved_samples.append((img, gt, pred_mask, res["iou"]))

        hog_val_results[key] = {
            "iou": round(np.mean(ious), 4),
            "f1": round(np.mean(f1s), 4),
            "precision": round(np.mean(pres), 4),
            "recall": round(np.mean(recs), 4)
        }
        plot_top_samples(saved_samples, f"HOG {clf_name} - {ds_name} VAL")

print_results("HOG FEATURE - VALIDATION RESULTS", hog_val_results)

#### 5.1.1.3 Use optimized trained model to predict on test dataset.

In [ ]:
hog_test_results = {}

for ds_name in ["EWS", "HLC", "LLC"]:
    if ds_name == "EWS":
        test_imgs = test_imgs_ews
        test_masks = test_masks_ews
    elif ds_name == "HLC":
        test_imgs = test_imgs_hlc
        test_masks = test_masks_hlc
    else:
        test_imgs = test_imgs_llc
        test_masks = test_masks_llc

    for key in hog_models:
        if not key.startswith(ds_name):
            continue

        model = hog_models[key]
        scaler = hog_scalers[key]

        ious = []
        f1s = []
        pres = []
        recs = []
        saved_samples = []

        for img, gt_mask in zip(test_imgs, test_masks):
            pred_mask = predict_image_hog(img, model, scaler)
            res = calculate_metrics(pred_mask, gt_mask)
            
            ious.append(res["iou"])
            f1s.append(res["f1"])
            pres.append(res["precision"])
            recs.append(res["recall"])
            saved_samples.append((img, gt, pred_mask, res["iou"]))

        hog_test_results[key] = {
            "iou": round(np.mean(ious), 4),
            "f1": round(np.mean(f1s), 4),
            "precision": round(np.mean(pres), 4),
            "recall": round(np.mean(recs), 4)
        }
        plot_top_samples(saved_samples, f"HOG {clf_name} - {ds_name} TEST")

print_results("HOG FEATURE - TEST RESULTS", hog_test_results)

#### 5.1.1.4 Conclusion: HOG Feature
- Best Settings
    - patch size=25, step=25
- Best Performance (Highest IoU)
    - LLC: rf_hog 0.7754
    - EWS: svm_hog / rf_hog 0.6585
    - HLC: rf_hog 0.6144
- Balance (Best precision-recall trade-off)
    - SVM is most balanced
    - RF shows competitive overall performance
    - KNN has relatively lower scores
- Robustness (Smallest fluctuation across datasets)
    - HOG is stable across all datasets: EWS(~0.66), HLC(~0.61), LLC(~0.77)
- Reasons
    - HOG effectively captures edge and gradient features of vegetation
    - KNN is more prone to overfitting on small patches
    - SVM and RF provide better generalization

### 5.1.2 Use LBP as the single feature and compare between classifiers.
#### 5.1.2.1 Define basic functions

In [ ]:
PATCH_SIZE_LBP = 25
STEP_LBP = 25

def extract_patches_lbp(img, mask):
    patches = []
    labels = []
    h, w = img.shape[:2]
    for y in range(0, h - PATCH_SIZE_LBP + 1, STEP_LBP):
        for x in range(0, w - PATCH_SIZE_LBP + 1, STEP_LBP):
            patch = img[y:y+PATCH_SIZE_LBP, x:x+PATCH_SIZE_LBP]
            patch_mask = mask[y:y+PATCH_SIZE_LBP, x:x+PATCH_SIZE_LBP]
            cy, cx = PATCH_SIZE_LBP // 2, PATCH_SIZE_LBP // 2
            label = patch_mask[cy, cx]
            patches.append(patch)
            labels.append(label)
    return patches, labels

def get_lbp_feature(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=59, range=(0, 58))
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-6)
    return hist

def build_dataset_lbp(imgs, masks):
    X = []
    y = []
    for img, mask in zip(imgs, masks):
        patches, labels = extract_patches_lbp(img, mask)
        for p, lb in zip(patches, labels):
            X.append(get_lbp_feature(p))
            y.append(lb)
    return np.array(X), np.array(y)

def predict_image_lbp(img, model, scaler):
    h, w = img.shape[:2]
    output = np.zeros((h, w), dtype=np.uint8)
    for y in range(0, h - PATCH_SIZE_LBP + 1, STEP_LBP):
        for x in range(0, w - PATCH_SIZE_LBP + 1, STEP_LBP):
            patch = img[y:y+PATCH_SIZE_LBP, x:x+PATCH_SIZE_LBP]
            feat = get_lbp_feature(patch).reshape(1, -1)
            feat = scaler.transform(feat)
            pred = model.predict(feat)[0]
            output[y:y+PATCH_SIZE_LBP, x:x+PATCH_SIZE_LBP] = pred
    return output

#### 5.1.2.2 Use train and validation dataset to train and optimize the model.

In [ ]:
lbp_models = {}
lbp_scalers = {}
lbp_val_results = {}

datasets = {
    "EWS": (train_imgs_ews, train_masks_ews, val_imgs_ews, val_masks_ews),
    "HLC": (train_imgs_hlc, train_masks_hlc, val_imgs_hlc, val_masks_hlc),
    "LLC": (train_imgs_llc, train_masks_llc, val_imgs_llc, val_masks_llc)
}

classifiers = {
    "svm": SVC(),
    "rf": RandomForestClassifier(n_estimators=50, random_state=SEED),
    "knn": KNeighborsClassifier(n_neighbors=3)
}

for ds_name in datasets:
    print(f"=== Training LBP on {ds_name} ===")
    tr_imgs, tr_masks, val_imgs, val_masks = datasets[ds_name]

    X_train, y_train = build_dataset_lbp(tr_imgs, tr_masks)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    for clf_name, clf in classifiers.items():
        print(f"Training {clf_name}...")
        clf.fit(X_train, y_train)

        key = f"{ds_name}_{clf_name}_lbp"
        lbp_models[key] = clf
        lbp_scalers[key] = scaler

        ious, f1s, pres, recs = [], [], [], []
        saved_samples = []
        for img, gt in zip(val_imgs, val_masks):
            pred_mask = predict_image_lbp(img, clf, scaler)
            res = calculate_metrics(pred_mask, gt)
            ious.append(res["iou"])
            f1s.append(res["f1"])
            pres.append(res["precision"])
            recs.append(res["recall"])
            saved_samples.append((img, gt, pred_mask, res["iou"]))

        lbp_val_results[key] = {
            "iou": round(np.mean(ious), 4),
            "f1": round(np.mean(f1s), 4),
            "precision": round(np.mean(pres), 4),
            "recall": round(np.mean(recs), 4)
        }
        plot_top_samples(saved_samples, f"LBP {clf_name} - {ds_name} VAL")

print_results("LBP FEATURE - VALIDATION RESULTS", lbp_val_results)

#### 5.1.2.3 Use optimized trained model to predict on test dataset.

In [ ]:
lbp_test_results = {}

for ds_name in ["EWS", "HLC", "LLC"]:
    if ds_name == "EWS":
        imgs = test_imgs_ews
        masks = test_masks_ews
    elif ds_name == "HLC":
        imgs = test_imgs_hlc
        masks = test_masks_hlc
    else:
        imgs = test_imgs_llc
        masks = test_masks_llc

    for key in lbp_models:
        if not key.startswith(ds_name):
            continue

        cl = lbp_models[key]
        sc = lbp_scalers[key]

        ious = []
        f1s = []
        pres = []
        recs = []
        saved_samples = []

        for img, gt in zip(imgs, masks):
            pred_mask = predict_image_lbp(img, cl, sc)
            res = calculate_metrics(pred_mask, gt)
            ious.append(res["iou"])
            f1s.append(res["f1"])
            pres.append(res["precision"])
            recs.append(res["recall"])
            saved_samples.append((img, gt, pred_mask, res["iou"]))

        lbp_test_results[key] = {
            "iou": round(np.mean(ious), 4),
            "f1": round(np.mean(f1s), 4),
            "precision": round(np.mean(pres), 4),
            "recall": round(np.mean(recs), 4)
        }
        plot_top_samples(saved_samples, f"LBP {clf_name} - {ds_name} TEST")

print_results("LBP FEATURE - TEST RESULTS", lbp_test_results)

#### 5.1.2.4 Conclusion: LBP Feature
- Best Settings
    - Optimal patch size = 25, step = 25.
- Best Performance
    - LLC: svm_lbp 0.7905
    - EWS: svm_lbp 0.6709
    - HLC: svm_lbp 0.6235
- Balance
    - SVM is the most balanced with the best trade-off between precision and recall.
    - RF maintains reasonable stability.
    - KNN shows obvious performance gap and imbalance.
- Robustness
    - SVM is the most robust with stable performance: EWS(0.6709), HLC(0.6235), LLC(0.7905).
    - KNN varies significantly and is the least robust.
- Reasons
    - SVM fits high-dimensional texture features stably with strong generalization.
    - RF is less sensitive to noise and maintains reliable prediction.
    - KNN overfits easily on local patches and performs worst on complex datasets.

### 5.1.3 Use GLCM as the single feature and compare between classifiers.
#### 5.1.3.1 Define basic functions

In [ ]:
PATCH_SIZE_GLCM = 25
STEP_GLCM = 25

def extract_patches_glcm(img, mask):
    patches = []
    labels = []
    h, w = img.shape[:2]
    for y in range(0, h - PATCH_SIZE_GLCM + 1, STEP_GLCM):
        for x in range(0, w - PATCH_SIZE_GLCM + 1, STEP_GLCM):
            patch = img[y:y+PATCH_SIZE_GLCM, x:x+PATCH_SIZE_GLCM]
            patch_mask = mask[y:y+PATCH_SIZE_GLCM, x:x+PATCH_SIZE_GLCM]
            cy, cx = PATCH_SIZE_GLCM // 2, PATCH_SIZE_GLCM // 2
            label = patch_mask[cy, cx]
            patches.append(patch)
            labels.append(label)
    return patches, labels

def get_glcm_feature(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    gray = (gray // 16).astype(np.uint8)
    
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=16, symmetric=True, normed=True)
    
    f1 = graycoprops(glcm, 'contrast')[0, 0]
    f2 = graycoprops(glcm, 'correlation')[0, 0]
    f3 = graycoprops(glcm, 'energy')[0, 0]
    f4 = graycoprops(glcm, 'homogeneity')[0, 0]
    
    return np.array([f1, f2, f3, f4])

def build_dataset_glcm(imgs, masks):
    X = []
    y = []
    for img, mask in zip(imgs, masks):
        patches, labels = extract_patches_glcm(img, mask)
        for p, lb in zip(patches, labels):
            X.append(get_glcm_feature(p))
            y.append(lb)
    return np.array(X), np.array(y)

def predict_image_glcm(img, model, scaler):
    h, w = img.shape[:2]
    output = np.zeros((h, w), dtype=np.uint8)
    for y in range(0, h - PATCH_SIZE_GLCM + 1, STEP_GLCM):
        for x in range(0, w - PATCH_SIZE_GLCM + 1, STEP_GLCM):
            patch = img[y:y+PATCH_SIZE_GLCM, x:x+PATCH_SIZE_GLCM]
            feat = get_glcm_feature(patch).reshape(1, -1)
            feat = scaler.transform(feat)
            pred = model.predict(feat)[0]
            output[y:y+PATCH_SIZE_GLCM, x:x+PATCH_SIZE_GLCM] = pred
    return output

#### 5.1.3.2 Use train and validation dataset to train and optimize the model.

In [ ]:
glcm_models = {}
glcm_scalers = {}
glcm_val_results = {}

datasets = {
    "EWS": (train_imgs_ews, train_masks_ews, val_imgs_ews, val_masks_ews),
    "HLC": (train_imgs_hlc, train_masks_hlc, val_imgs_hlc, val_masks_hlc),
    "LLC": (train_imgs_llc, train_masks_llc, val_imgs_llc, val_masks_llc)
}

classifiers = {
    "svm": SVC(),
    "rf": RandomForestClassifier(n_estimators=50, random_state=SEED),
    "knn": KNeighborsClassifier(n_neighbors=3)
}

for ds_name in datasets:
    print(f"=== Training GLCM on {ds_name} ===")
    tr_imgs, tr_masks, val_imgs, val_masks = datasets[ds_name]

    X_train, y_train = build_dataset_glcm(tr_imgs, tr_masks)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    for clf_name, clf in classifiers.items():
        print(f"Training {clf_name}...")
        clf.fit(X_train, y_train)

        key = f"{ds_name}_{clf_name}_glcm"
        glcm_models[key] = clf
        glcm_scalers[key] = scaler

        ious, f1s, pres, recs = [], [], [], []
        saved_samples = []
        for img, gt in zip(val_imgs, val_masks):
            pred_mask = predict_image_glcm(img, clf, scaler)
            res = calculate_metrics(pred_mask, gt)
            ious.append(res["iou"])
            f1s.append(res["f1"])
            pres.append(res["precision"])
            recs.append(res["recall"])
            saved_samples.append((img, gt, pred_mask, res["iou"]))

        glcm_val_results[key] = {
            "iou": round(np.mean(ious), 4),
            "f1": round(np.mean(f1s), 4),
            "precision": round(np.mean(pres), 4),
            "recall": round(np.mean(recs), 4)
        }
        plot_top_samples(saved_samples, f"GLCM {clf_name} - {ds_name} VAL")

print_results("GLCM FEATURE - VALIDATION RESULTS", glcm_val_results)

#### 5.1.3.3 Use optimized trained model to predict on test dataset.

In [ ]:
glcm_test_results = {}

for ds_name in ["EWS", "HLC", "LLC"]:
    if ds_name == "EWS":
        imgs = test_imgs_ews
        masks = test_masks_ews
    elif ds_name == "HLC":
        imgs = test_imgs_hlc
        masks = test_masks_hlc
    else:
        imgs = test_imgs_llc
        masks = test_masks_llc

    for key in glcm_models:
        if not key.startswith(ds_name):
            continue

        cl = glcm_models[key]
        sc = glcm_scalers[key]

        ious = []
        f1s = []
        pres = []
        recs = []
        saved_samples = []

        for img, gt in zip(imgs, masks):
            pred_mask = predict_image_glcm(img, cl, sc)
            res = calculate_metrics(pred_mask, gt)
            ious.append(res["iou"])
            f1s.append(res["f1"])
            pres.append(res["precision"])
            recs.append(res["recall"])
            saved_samples.append((img, gt, pred_mask, res["iou"]))

        glcm_test_results[key] = {
            "iou": round(np.mean(ious), 4),
            "f1": round(np.mean(f1s), 4),
            "precision": round(np.mean(pres), 4),
            "recall": round(np.mean(recs), 4)
        }
        plot_top_samples(saved_samples, f"GLCM {clf_name} - {ds_name} TEST")

print_results("GLCM FEATURE - TEST RESULTS", glcm_test_results)

#### 5.1.3.4 Conclusion: GLCM Feature
- Best Settings
    - Optimal patch size = 25, step = 25.
- Best Performance
    - LLC: svm_glcm 0.7907
    - EWS: svm_glcm 0.6705
    - HLC: svm_glcm 0.6230
- Balance
    - SVM achieves extremely high recall but shows low precision, leading to imbalance.
    - RF provides better precision-recall coordination.
    - KNN has the lowest overall scores and poorest balance.
- Robustness
    - SVM is the most stable across datasets with minor IoU changes.
    - RF and KNN exhibit larger performance drops.
- Reasons
    - GLCM is a low-dimensional statistical feature; SVM tends to predict positive samples excessively.
    - RF reduces bias and improves prediction balance.
    - KNN lacks discriminative power on simple statistical features, resulting in low accuracy.   
#### 5.1.4 Conclusion: Single Features
- Best Performance
    - Overall best: svm_glcm on LLC with IoU = 0.7907.
    - Second best: svm_lbp on LLC with IoU = 0.7905.
- Balance
    - svm_lbp delivers the most consistent precision-recall balance across all datasets.
    - svm_hog achieves good and stable trade-off.
    - svm_glcm has extremely high recall but is highly imbalanced.
    - All KNN models show weak balance and stability.
- Robustness
    - svm_lbp is the most robust with the smallest cross-dataset fluctuation.
    - svm_hog is the second most robust.
    - svm_glcm achieves high IoU but lacks balance.
    - KNN models are the least robust and unstable.
- Reasons
    - LBP captures local detailed texture and is robust to vegetation variations.
    - HOG captures structural gradient information with good stability.
    - GLCM uses simple global statistics and leads to biased predictions.
    - KNN relies on distance matching and overfits easily on small patches.


## 5.2 Use combined features to train the model
Since the optimized patch size and step for all the features are 50. We use 50 as the fixed patch size and step size.

### 5.2.1 Define basic functions

In [ ]:
PATCH_SIZE = 25
STEP_SIZE = 25

def hog_f(patch):
    if len(patch.shape) == 3:
        gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        gray = patch.copy()
    return hog(gray, pixels_per_cell=(8,8), cells_per_block=(2,2), visualize=False)

def lbp_f(patch):
    if len(patch.shape) == 3:
        gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        gray = patch.copy()
    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=59, range=(0,58))
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-6)
    return hist

def glcm_f(patch):
    if len(patch.shape) == 3:
        gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        gray = patch.copy()
    gray = (gray // 16).astype(np.uint8)
    g = graycomatrix(gray, [1], [0], 16, symmetric=True, normed=True)
    return np.array([
        graycoprops(g, 'contrast')[0,0],
        graycoprops(g, 'correlation')[0,0],
        graycoprops(g, 'energy')[0,0],
        graycoprops(g, 'homogeneity')[0,0]
    ])

combinations = {
    "hog_lbp": lambda p: np.concatenate([hog_f(p), lbp_f(p)]),
    "hog_glcm": lambda p: np.concatenate([hog_f(p), glcm_f(p)]),
    "lbp_glcm": lambda p: np.concatenate([lbp_f(p), glcm_f(p)]),
    "hog_lbp_glcm": lambda p: np.concatenate([hog_f(p), lbp_f(p), glcm_f(p)])
}

def extract_patches(img, mask):
    patches, labels = [], []
    h, w = img.shape[:2]
    for y in range(0, h-PATCH_SIZE+1, STEP_SIZE):
        for x in range(0, w-PATCH_SIZE+1, STEP_SIZE):
            p = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            m = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            patches.append(p)
            labels.append(m[PATCH_SIZE//2, PATCH_SIZE//2])
    return patches, labels

def build_dataset(imgs, masks, feat_func):
    X, y = [], []
    for img, mask in zip(imgs, masks):
        ps, lbs = extract_patches(img, mask)
        for p, lb in zip(ps, lbs):
            X.append(feat_func(p))
            y.append(lb)
    return np.array(X), np.array(y)

def predict_image(img, model, scaler, feat_func):
    h, w = img.shape[:2]
    out = np.zeros((h, w), dtype=np.uint8)
    for y in range(0, h-PATCH_SIZE+1, STEP_SIZE):
        for x in range(0, w-PATCH_SIZE+1, STEP_SIZE):
            p = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            f = feat_func(p).reshape(1,-1)
            f = scaler.transform(f)
            out[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = model.predict(f)[0]
    return out

### 5.2.2 Use train and validation dataset to train and optimize the model.

In [ ]:
combo_models = {}
combo_scalers = {}
combo_val_results = {}

datasets = {
    "EWS": (train_imgs_ews, train_masks_ews, val_imgs_ews, val_masks_ews),
    "HLC": (train_imgs_hlc, train_masks_hlc, val_imgs_hlc, val_masks_hlc),
    "LLC": (train_imgs_llc, train_masks_llc, val_imgs_llc, val_masks_llc)
}

for ds_name in datasets:
    print(f'Train on {ds_name}')
    tr_imgs, tr_masks, val_imgs, val_masks = datasets[ds_name]
    
    for cname, feat in combinations.items():
        print(f'  with feature {cname}')
        X, y = build_dataset(tr_imgs, tr_masks, feat)
        sc = StandardScaler()
        X = sc.fit_transform(X)
        
        # Must construct new classifiers every time to avoid leaking.
        clf_dict = {
            "svm": SVC(),
            "rf": RandomForestClassifier(n_estimators=50, random_state=SEED),
            "knn": KNeighborsClassifier(n_neighbors=3)
        }
        
        for clf_name, clf in clf_dict.items():
            print(f'    with classifier {clf_name}')
            clf.fit(X, y)
            key = f"{ds_name}_{clf_name}_{cname}"
            combo_models[key] = clf
            combo_scalers[key] = sc
            
            ious, f1s, pres, recs = [], [], [], []
            saved_samples = []
            for img, gt in zip(val_imgs, val_masks):
                pred = predict_image(img, clf, sc, feat)
                res = calculate_metrics(pred, gt)
                ious.append(res["iou"])
                f1s.append(res["f1"])
                pres.append(res["precision"])
                recs.append(res["recall"])
                saved_samples.append((img, gt, pred, res["iou"]))
            
            combo_val_results[key] = {
                "iou": round(np.mean(ious),4),
                "f1": round(np.mean(f1s),4),
                "precision": round(np.mean(pres),4),
                "recall": round(np.mean(recs),4)
            }
            plot_top_samples(saved_samples, f"COMBINED {clf_name} - {ds_name} VAL")

print_results("COMBINED FEATURE - VALIDATION RESULTS", combo_val_results)

### 5.2.3 Use optimized trained model to predict on test dataset.

In [ ]:
combo_test_results = {}

for ds_name in ["EWS","HLC","LLC"]:
    if ds_name == "EWS":
        imgs, masks = test_imgs_ews, test_masks_ews
    elif ds_name == "HLC":
        imgs, masks = test_imgs_hlc, test_masks_hlc
    else:
        imgs, masks = test_imgs_llc, test_masks_llc

    for cname, feat in combinations.items():
        print(f'Predict with feature {cname}')
        for clf_name in ["svm","rf","knn"]:
            key = f"{ds_name}_{clf_name}_{cname}"
            if key not in combo_models:
                continue
            print(f'  on {ds_name} with classifier {clf_name}')
            cl = combo_models[key]
            sc = combo_scalers[key]
            
            ious, f1s, pres, recs = [], [], [], []
            saved_samples = []
            for img, gt in zip(imgs, masks):
                pred = predict_image(img, cl, sc, feat)
                res = calculate_metrics(pred, gt)
                ious.append(res["iou"])
                f1s.append(res["f1"])
                pres.append(res["precision"])
                recs.append(res["recall"])
                saved_samples.append((img, gt, pred, res["iou"]))
            
            combo_test_results[key] = {
                "iou": round(np.mean(ious),4),
                "f1": round(np.mean(f1s),4),
                "precision": round(np.mean(pres),4),
                "recall": round(np.mean(recs),4)
            }
            plot_top_samples(saved_samples, f"COMBINED {clf_name} - {ds_name} TEST")

print_results("COMBINED FEATURE - TEST RESULTS", combo_test_results)

### 5.2.4 Conclusion: Combined Features
- Best Combination
  - Overall best: svm_lbp_glcm
    - EWS: 0.6735
    - HLC: 0.6258
    - LLC: 0.7915
- Balance
  - lbp_glcm provides the best balance between precision and recall.
  - Adding HOG to form triple combinations does not improve balance.
- Robustness
  - lbp_glcm shows the smallest cross-dataset fluctuation and is the most robust.
  - The triple-combination feature suffers from redundancy and slightly reduced generalization.
  - KNN is unstable across all combined features.
- Reasons
  - LBP (local texture) and GLCM (global texture) complement each other with low redundancy.
  - HOG shares gradient information with LBP, leading to diminishing gains in triple features.
  - KNN overfits on high-dimensional combined features, resulting in weak performance, consistent with observations.
## 5.3 Conclusion: Single Features vs Combined Features
- Best Performance
  - Combined features significantly outperform single features: svm_lbp_glcm achieves 0.7915 on LLC, higher than the best single feature svm_glcm (0.7907).
  - Combined features also lead on EWS and HLC: svm_lbp_glcm achieves 0.6735 on EWS and 0.6258 on HLC, which are higher than the best single features on the two datasets (svm_lbp 0.6709 on EWS, svm_lbp 0.6235 on HLC).
- Balance
  - Combined features show a more balanced precision-recall trade-off without extreme bias.
  - Among single features, GLCM has inflated recall; LBP and HOG are more balanced but overall weaker than combined features.
- Robustness
  - Combined features are much more stable across datasets with smaller performance fluctuations.
  - Single features rely on limited information and are more sensitive to data distribution changes.
- Reasons
  - Combined features fuse multi-scale and multi-type information for stronger representation.
  - Single features only capture one attribute and lack sufficient information for complex lesion regions.
  - Feature complementarity effectively improves generalization, making combined features superior in optimality, balance, and robustness.